# Baseline for dialogue tuning

In [ ]:
%load_ext autoreload
%autoreload 2
import re, json, numpy as np, torch
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
from gpt.generate import load_model, generate_batch

RUN = "../runs/sft_lr1e-4"                    # the Shakespeare SFT model
TAG = "baseline_sft_lr1e-4"                   # name of the saved results file
tok = Tokenizer.from_file("../data_sft/tokenizer.json")
model, cfg = load_model(f"{RUN}/best.pt")
from gpt.utils import check_tokenizer, checkpoint_tokenizer_hash
check_tokenizer(f"../data_sft/tokenizer.json", cfg.tokenizer_sha256 or checkpoint_tokenizer_hash(f"{RUN}/best.pt"))
eot_id = tok.token_to_id("<|endoftext|>")

# how do blank lines tokenize? (matters for choosing a stop token later)
for s in ["\n\n", "line.\n\nJULIET:\n"]:
    ids = tok.encode(s).ids
    print(repr(s), ids, [tok.id_to_token(i) for i in ids])

raw = open("../data_sft/raw/input.txt", encoding="utf-8").read()
speakers = Counter(re.findall(r"^([A-Za-z' ]+):$", raw, flags=re.M))

# (history, partner): hand-written lines, mixing verse, modern English, and one multi-turn context
CONTEXTS = [
    ([("ROMEO", "Wilt thou not walk with me beneath the moon?")], "JULIET"),
    ([("ROMEO", "Hey, I think I left my keys in your garden.")], "JULIET"),
    ([("PETRUCHIO", "Good morrow, sweet Kate! Hast thou seen my hat?")], "KATHARINA"),
    ([("LUCIO", "Come, sir, a jest! the duke is gone to sea.")], "ISABELLA"),
    ([("MENENIUS", "The people murmur, and the corn grows scarce.")], "SICINIUS"),
    ([("GLOUCESTER", "Our brother sleeps; let us not wake the court.")], "QUEEN ELIZABETH"),
    ([("ROMEO", "I have a gift for thee."), ("JULIET", "A gift? What manner of gift?"),
      ("ROMEO", "A boat, that we may sail away tonight.")], "JULIET"),
    ([("LEONTES", "Thou hast a dragon in thy cellar, friend!")], "PAULINA"),
]
missing = sorted({p for _, p in CONTEXTS if p not in speakers})
if missing:
    print("warning: partner names not in the corpus (replies will be odd):", missing)

SPLIT = re.compile(r"\n\s*\n|\n(?=[A-Z][A-Za-z' ]+:\s*\n)")        # blank line, or the next speaker tag
STOPW = {"the", "a", "an", "and", "or", "but", "to", "of", "in", "on", "at", "is", "are", "was", "i", "you", "he",
         "she", "it", "my", "thy", "thou", "thee", "that", "this", "with", "for", "not", "have", "hast", "will"}
words = lambda s: re.findall(r"[a-z']+", s.lower())
content = lambda s: {w for w in words(s) if w not in STOPW and len(w) > 2}


@torch.no_grad()
def sample_replies(history, partner, n=8, max_new_tokens=60, temperature=1.0, k=None, seed=0):
    text = "\n\n".join(f"{s}:\n{l}" for s, l in history) + f"\n\n{partner}:\n"
    ids = ([eot_id] + tok.encode(text).ids)[-(cfg.context_len - max_new_tokens):]
    idx = torch.tensor([ids] * n)                                     # one context, replicated n times: equal lengths, no padding
    g = torch.Generator().manual_seed(seed)
    out, finished, n_new = generate_batch(model, cfg.context_len, idx, max_new_tokens, temperature, k, g, [eot_id], eot_id)
    T0, replies = idx.shape[1], []
    for b in range(n):
        new = out[b, T0:T0 + int(n_new[b])].tolist()
        if eot_id in new:
            new = new[:new.index(eot_id)]                              # the document ended
        parts = SPLIT.split(tok.decode(new), maxsplit=1)
        replies.append({"reply": parts[0].strip(), "stopped": len(parts) > 1})
    return replies


def repeat_bigram(w):
    return 0.0 if len(w) < 2 else 1 - len(set(zip(w, w[1:]))) / (len(w) - 1)


raw_counts = Counter(words(raw))
SHAKE = set(raw_counts)                                              # any word that occurs in the plays

story_ids = np.memmap("../data/val.bin", dtype=np.uint16, mode="r")[:2_000_000]    # TinyStories tokenized with the same tokenizer
story_counts = Counter(words(tok.decode(story_ids.tolist())))
STORY = {w for w, c in story_counts.items() if c >= 5}              # common children's-story words


def word_class(w):
    return "shake" if w in SHAKE else ("story" if w in STORY else "invented")


def class_rates(ws):
    cs = [word_class(w) for w in ws] or ["invented"]
    return {c: cs.count(c) / len(cs) for c in ("shake", "story", "invented")}


def metrics(prev_line, replies):
    r = [x["reply"] for x in replies]
    w = [words(x) for x in r]
    c_prev = content(prev_line)
    out = {
        "stopped": float(np.mean([x["stopped"] for x in replies])),
        "words": float(np.mean([len(x) for x in w])),
        "overlap": float(np.mean([len(content(x) & c_prev) for x in r])),
        "new_words": float(np.mean([len(content(x) - c_prev) for x in r])),
        "repeat_bigram": float(np.mean([repeat_bigram(x) for x in w])),
        "distinct_replies": len(set(r)) / len(r),
    }
    rates = [class_rates(x) for x in w]
    for c in ("shake", "story", "invented"):
        out[f"{c}_words"] = float(np.mean([q[c] for q in rates]))
    return out


rows = []
for history, partner in CONTEXTS:
    reps = sample_replies(history, partner)
    m = {k: float(v) for k, v in metrics(history[-1][1], reps).items()}
    rows.append({"history": history, "partner": partner, "replies": reps, "metrics": m})
    print("=" * 78)
    print(" | ".join(f"{s}: {l}" for s, l in history), f"\n--> {partner}")
    for x in reps[:4]:
        print("  •", x["reply"].replace("\n", " / ") or "(empty)")
    print("  ", {k: round(v, 2) for k, v in m.items()})

mean = {k: round(float(np.mean([r["metrics"][k] for r in rows])), 3) for k in rows[0]["metrics"]}
print("\nMEAN OVER CONTEXTS:", mean)

Path("../runs/improv").mkdir(parents=True, exist_ok=True)
json.dump({"run": RUN, "sampling": {"temperature": 1.0, "k": None, "n": 8, "max_new_tokens": 60}, "mean": mean, "rows": rows},
          open(f"../runs/improv/{TAG}.json", "w"), indent=1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
'\n\n' [201, 201] ['Ċ', 'Ċ']
'line.\n\nJULIET:\n' [78, 807, 16, 201, 201, 44, 55, 46, 43, 39, 54, 28, 201] ['l', 'ine', '.', 'Ċ', 'Ċ', 'J', 'U', 'L', 'I', 'E', 'T', ':', 'Ċ']
ROMEO: Wilt thou not walk with me beneath the moon? 
--> JULIET
  • Squeak, my lord! / Such root! leave me with whom I talk: what, my lord?
  • I have heard a single man durst not; may but let his / kestest worship.
  • He's not a carfellow that's a chamberword. / Come, let's to London at his level, / It's irrely command'd the field, / And be not go into ghostes or drornms too wrong
  • I: / True, mate John, that's my prison.
   {'stopped': 0.62, 'words': 19.88, 'overlap': 0.12, 'new_words': 11.62, 'repeat_bigram': 0.02, 'distinct_replies': 1.0, 'shake_words': 0.92, 'story_words': 0.0, 'invented_words': 0.08}
ROMEO: Hey, I think I left my keys in your garden. 
--> JULIET
  • What? 'tis disvolt to have credit pronounce?
  • I do

In [5]:
for row in rows[:4]:
    inv = [x for r in row["replies"] for x in words(r["reply"]) if word_class(x) == "invented"]
    print(row["partner"], "->", sorted(set(inv))[:25])

JULIET -> ['ably', 'carfellow', 'chamberword', "command'd", 'disme', "drift'd", 'drornms', 'fastiar', 'ghostes', 'irrely', 'kestest', "lament'd", 'leoted', 'mesterce', 'rector', 'solef', 'squeak', 'undert']
JULIET -> ['applate', 'disvolt', 'nameor', 'rickions', 'sents', 'vessations']
KATHARINA -> ['blary', 'chuke', 'crayed', 'grandchur']
ISABELLA -> ['dels', 'dickone', 'doft', 'fpl', 'pieet', 'rure', 'traites', 'wrutiable']


In [9]:
import re
from pathlib import Path

from gpt.improv_data import split_turn

def load_turns(path):
    text = Path(path).read_text(encoding="utf-8")
    return [
        t.strip()
        for t in re.split(r"\n\s*\n", text)
        if t.strip()
    ]

def is_speaker_turn(turn):
    tag, body = split_turn(turn)
    return bool(re.fullmatch(r"[A-Za-z][A-Za-z0-9 .,'&-]*:", tag))

turns = load_turns("../data_sft/raw/input.txt")

print("Total turns:", len(turns))
print("First 20:")
for t in turns[:20]:
    print(repr(t.split("\n")[0]))

print("Invalid turns:", sum(not is_speaker_turn(t) for t in turns))

Total turns: 7222
First 20:
'First Citizen:'
'All:'
'First Citizen:'
'All:'
'First Citizen:'
'All:'
'First Citizen:'
'All:'
'Second Citizen:'
'First Citizen:'
'Second Citizen:'
'All:'
'Second Citizen:'
'First Citizen:'
'Second Citizen:'
'First Citizen:'
'Second Citizen:'
'First Citizen:'
'All:'
'First Citizen:'
Invalid turns: 0


"Senators, &C:\nWe'll surety him."
---
"Senators, &C:\nWeapons, weapons, weapons!\n'Tribunes!' 'Patricians!' 'Citizens!' 'What, ho!'\n'Sicinius!' 'Brutus!' 'Coriolanus!' 'Citizens!'\n'Peace, peace, peace!' 'Stay, hold, peace!'"
---


In [ ]:
for t in turns:
    if not is_speaker_turn(t):
        print(repr(t[:300]))
        print("---")

In [12]:
def get_speakers(turns):
    return sorted({
        split_turn(turn)[0]
        for turn in turns
        if is_speaker_turn(turn)
    })

print("Total turns:", len(turns))
print("Invalid turns:", sum(not is_speaker_turn(t) for t in turns))

print("\nDistinct speakers:")
print(get_speakers(turns))

print("\nPreviously invalid turns:")
for t in turns:
    if "Senators, &C:" in t:
        print(repr(t[:150]))

Total turns: 7222
Invalid turns: 0

Distinct speakers:
['A Patrician:', 'A Player:', 'ABHORSON:', 'ABRAHAM:', 'ADRIAN:', 'AEdile:', 'ALL SERVING-MEN:', 'ALL:', 'ALONSO:', 'ANGELO:', 'ANOTHER:', 'ANTIGONUS:', 'ANTONIO:', 'ARCHBISHOP OF YORK:', 'ARCHIDAMUS:', 'ARIEL:', 'AUFIDIUS:', 'AUTOLYCUS:', 'Abbot:', 'All Citizens:', 'All Conspirators:', 'All The Lords:', 'All The People:', 'All:', 'Apothecary:', 'BAGOT:', 'BALTHASAR:', 'BAPTISTA:', 'BARNARDINE:', 'BENVOLIO:', 'BIANCA:', 'BIONDELLO:', 'BISHOP OF CARLISLE:', 'BISHOP OF ELY:', 'BLUNT:', 'BONA:', 'BRAKENBURY:', 'BRUTUS:', 'BUCKINGHAM:', 'BUSHY:', 'Boatswain:', 'Both Citizens:', 'Both Tribunes:', 'Both:', 'Boy:', 'CALIBAN:', 'CAMILLO:', 'CAPULET:', 'CARDINAL:', 'CATESBY:', 'CHRISTOPHER:', 'CLARENCE:', 'CLAUDIO:', 'CLEOMENES:', 'CLIFFORD:', 'COMINIUS:', 'CORIOLANUS:', 'CURTIS:', 'Captain:', 'Children:', 'Chorus:', 'Citizen:', 'Citizens:', 'Clown:', 'DERBY:', 'DION:', 'DORCAS:', 'DORSET:', 'DUCHESS OF YORK:', 'DUCHESS:', 'DUKE OF AUMERLE:

## Dry run

In [14]:
import sys, json, random
import numpy as np, pandas as pd
from pathlib import Path
from tokenizers import Tokenizer

ROOT = Path("..").resolve()                                   # drop this if the notebook is in the repo root
sys.path.insert(0, str(ROOT / "scripts"))
from scripts.prep_sft import parse_turns                           # or from gpt.improv_data once you move it
from gpt.improv_data import build_improv_examples, split_turn

tok = Tokenizer.from_file(str(ROOT / "data_sft/tokenizer.json"))
eot_id, eoturn_id = tok.token_to_id("<|endoftext|>"), tok.token_to_id("<|endofturn|>")
print("ids:", eot_id, eoturn_id, "| vocab:", tok.get_vocab_size())
turns = parse_turns((ROOT / "data_sft/raw/input.txt").read_text(encoding="utf-8"))
train, val, stats = build_improv_examples(turns, tok, eot_id=eot_id, eoturn_id=eoturn_id, seed=42)
print(json.dumps(stats, indent=1, default=str))
assert train and val, "an empty split"

# structural invariants on the real data
for name, exs in (("train", train), ("val", val)):
    for e in exs:
        assert e["ids"][0] == eot_id and e["ids"][-1] == eoturn_id
        assert e["total_len"] <= 256 and e["total_len"] - e["prompt_len"] >= 2     # at least one reply token plus the end-of-turn
        assert all(i // 50 == e["block_index"] for i in e["context_indices"] + [e["target_index"]]), "context crosses a block"
        assert ((e["block_index"] + 1) % 10 == 0) == (name == "val"), "block in the wrong split"
print("invariants OK")


def summarize(exs, name):
    df = pd.DataFrame({
        "reply_tokens": [e["total_len"] - e["prompt_len"] - 1 for e in exs],
        "total_len": [e["total_len"] for e in exs],
        "n_context": [len(e["context_indices"]) for e in exs],
        "user": [e["user_speaker"] for e in exs],
        "target": [split_turn(turns[e["target_index"]])[0] for e in exs],
        "episode": [(e["block_index"], e["episode_index"]) for e in exs],
    })
    print(f"\n=== {name}: {len(df)} examples, {df.episode.nunique()} episodes, "
          f"{df.target.nunique()} target speakers, {len({e['target_index'] for e in exs})} distinct target turns")
    print(df[["reply_tokens", "total_len"]].describe().round(1).T)
    print("replies of <=3 tokens:", f"{(df.reply_tokens <= 3).mean():.1%}")
    print("context length:", df.n_context.value_counts().sort_index().to_dict())
    print("top targets:", (df.target.value_counts(normalize=True).head(8).round(3)).to_dict())
    print("top users:  ", (df.user.value_counts(normalize=True).head(8).round(3)).to_dict())
    print("examples per episode:", df.groupby("episode").size().describe().round(1).to_dict())

summarize(train, "train"); summarize(val, "val")

# read some real examples
random.seed(0)
for e in random.sample(train, 8):
    print("-" * 70, f"\nuser={e['user_speaker']} | context turns={len(e['context_indices'])}")
    print(tok.decode(e["ids"][1:e["prompt_len"]]))
    print(">>>", tok.decode(e["ids"][e["prompt_len"]:-1]))

ModuleNotFoundError: No module named 'scripts'

In [25]:
%load_ext autoreload
%autoreload 2
import re, json, numpy as np, torch
from pathlib import Path
from collections import Counter
from tokenizers import Tokenizer
from gpt.generate import load_model, generate_batch

RUN = "../runs/sft_blocks_lr1e-4"
DATA = "../data_sft_blocks"
TAG = "baseline_blocks"
tok = Tokenizer.from_file(f"{DATA}/tokenizer.json")
model, cfg = load_model(f"{RUN}/best.pt")
from gpt.utils import check_tokenizer, checkpoint_tokenizer_hash
check_tokenizer(f"{DATA}/tokenizer.json", cfg.tokenizer_sha256 or checkpoint_tokenizer_hash(f"{RUN}/best.pt"))
eot_id = tok.token_to_id("<|endoftext|>")

raw = open(f"{DATA}/raw/input.txt", encoding="utf-8").read()
speakers = Counter(re.findall(r"^([A-Za-z' ]+):$", raw, flags=re.M))

# ---- words and vocabularies ----
words = lambda s: [w.strip("'") for w in re.findall(r"[a-z']+", s.lower()) if w.strip("'")]
STOPW = {"the", "a", "an", "and", "or", "but", "to", "of", "in", "on", "at", "is", "are", "was", "i", "you", "he",
         "she", "it", "my", "thy", "thou", "thee", "that", "this", "with", "for", "not", "have", "hast", "will"}
content = lambda s: {w for w in words(s) if w not in STOPW and len(w) > 2}

SHAKE = set(words(raw))                                              # any word in the plays
from datasets import load_dataset
STORY_FILE = Path("../data_sft_blocks/story_vocab.json")          # cached, so the stream runs once
if STORY_FILE.exists():
    STORY = set(json.load(open(STORY_FILE)))
else:
    counts = Counter()
    for i, ex in enumerate(load_dataset("roneneldan/TinyStories", split="validation", streaming=True)):
        counts.update(words(ex["text"]))
        if i >= 20_000:
            break
    STORY = {w for w, c in counts.items() if c >= 5}
    json.dump(sorted(STORY), open(STORY_FILE, "w"))
print("story vocabulary:", len(STORY), "| SHAKE:", len(SHAKE))
print([w for w in ["escaping", "believes", "gears", "pried", "later", "ponder"] if w in STORY])    # sanity check


def word_class(w):
    return "shake" if w in SHAKE else ("story" if w in STORY else "invented")


def class_rates(ws):
    cs = [word_class(w) for w in ws] or ["invented"]
    return {c: cs.count(c) / len(cs) for c in ("shake", "story", "invented")}


def repeat_bigram(w):
    return 0.0 if len(w) < 2 else 1 - len(set(zip(w, w[1:]))) / (len(w) - 1)


# ---- contexts: (history, partner) ----
CONTEXTS = [
    ([("ROMEO", "Wilt thou not walk with me beneath the moon?")], "JULIET"),
    ([("ROMEO", "Hey, I think I left my keys in your garden.")], "JULIET"),
    ([("PETRUCHIO", "Good morrow, sweet Kate! Hast thou seen my hat?")], "KATHARINA"),
    ([("LUCIO", "Come, sir, a jest! the duke is gone to sea.")], "ISABELLA"),
    ([("MENENIUS", "The people murmur, and the corn grows scarce.")], "SICINIUS"),
    ([("GLOUCESTER", "Our brother sleeps; let us not wake the court.")], "QUEEN ELIZABETH"),
    ([("ROMEO", "I have a gift for thee."), ("JULIET", "A gift? What manner of gift?"),
      ("ROMEO", "A boat, that we may sail away tonight.")], "JULIET"),
    ([("LEONTES", "Thou hast a dragon in thy cellar, friend!")], "PAULINA"),
]
missing = sorted({p for _, p in CONTEXTS if p not in speakers})
if missing:
    print("warning: partner names not in the corpus:", missing)

SPLIT = re.compile(r"\n\s*\n|\n(?=[A-Z][A-Za-z' ]+:\s*\n)")        # blank line, or the next speaker tag


@torch.no_grad()
def sample_replies(history, partner, n=8, max_new_tokens=60, temperature=1.0, k=None, seed=0):
    text = "\n\n".join(f"{s}:\n{l}" for s, l in history) + f"\n\n{partner}:\n"
    ids = ([eot_id] + tok.encode(text).ids)[-(cfg.context_len - max_new_tokens):]
    idx = torch.tensor([ids] * n)                                     # one context replicated n times: equal lengths
    g = torch.Generator().manual_seed(seed)
    out, finished, n_new = generate_batch(model, cfg.context_len, idx, max_new_tokens, temperature, k, g, [eot_id], eot_id)
    T0, replies = idx.shape[1], []
    for b in range(n):
        new = out[b, T0:T0 + int(n_new[b])].tolist()
        if eot_id in new:
            new = new[:new.index(eot_id)]
        parts = SPLIT.split(tok.decode(new), maxsplit=1)
        replies.append({"reply": parts[0].strip(), "stopped": len(parts) > 1})
    return replies

def distinct_n(replies, n=2):                         # unique n-grams across all replies of one context
    grams = [g for r in replies for g in zip(*[words(r)[i:] for i in range(n)])]
    return len(set(grams)) / max(1, len(grams))

def metrics(prev_line, replies):
    r = [x["reply"] for x in replies]
    w = [words(x) for x in r]
    c_prev = content(prev_line)
    out = {
        "stopped": float(np.mean([x["stopped"] for x in replies])),
        "words": float(np.mean([len(x) for x in w])),
        "overlap": float(np.mean([len(content(x) & c_prev) for x in r])),
        "new_words": float(np.mean([len(content(x) - c_prev) for x in r])),
        "repeat_bigram": float(np.mean([repeat_bigram(x) for x in w])),
        "distinct_replies": len(set(r)) / len(r),
        "overlap_frac": float(np.mean([len(content(x) & c_prev) / max(1, len(content(x))) for x in r])),
    }
    rates = [class_rates(x) for x in w]
    for c in ("shake", "story", "invented"):
        out[f"{c}_words"] = float(np.mean([q[c] for q in rates]))
    return out


rows = []
for history, partner in CONTEXTS:
    reps = sample_replies(history, partner)
    m = metrics(history[-1][1], reps)
    rows.append({"history": history, "partner": partner, "replies": reps, "metrics": m})
    print("=" * 78)
    print(" | ".join(f"{s}: {l}" for s, l in history), f"\n--> {partner}")
    for x in reps[:4]:
        print("  •", x["reply"].replace("\n", " / ") or "(empty)")
    print("  ", {k: round(v, 2) for k, v in m.items()})

mean = {k: round(float(np.mean([r["metrics"][k] for r in rows])), 3) for k in rows[0]["metrics"]}
print("\nMEAN OVER CONTEXTS:", mean)

for row in rows[:4]:                                                  # which words count as "invented"?
    inv = [x for r in row["replies"] for x in words(r["reply"]) if word_class(x) == "invented"]
    print(row["partner"], "->", sorted(set(inv))[:25])

Path("../runs/improv").mkdir(parents=True, exist_ok=True)
json.dump({"run": RUN, "sampling": {"temperature": 1.0, "k": None, "n": 8, "max_new_tokens": 60}, "mean": mean, "rows": rows},
          open(f"../runs/improv/{TAG}.json", "w"), indent=1)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
story vocabulary: 6683 | SHAKE: 12373
['gears', 'later']
ROMEO: Wilt thou not walk with me beneath the moon? 
--> JULIET
  • Swall thy that raging thy bark, / Yet, marry, say that it is a wooish whore: / I am now two hap banished, and ere I make her dead, / From gallawing liberty, and
  • I am a mystery, I cannot betwhat's against him.
  • Very well to meet in the porining of the stars / Sure conclus your grace's content lord as death / That pray thus to I veot. I wish thus short thread / To meet God R danger! But that die langu
  • I ponder of it: by what weare, what, a kind, that puts / As I dismX shall be upon, the most dealen than / That in what time may go out, but in his escaping visits / Although he unw
   {'stopped': 0.5, 'words': 16.88, 'overlap': 0.0, 'new_words': 9.5, 'repeat_bigram': 0.0, 'distinct_replies': 1.0, 'overlap_frac': 0.0, 'shake_words': 0.92, 'story_words': 0.0, 'invented_wor

In [26]:
print("RUN:", RUN, "| TAG:", TAG)
print("story vocab size:", len(STORY), "| SHAKE:", len(SHAKE))
print([w for w in ["escaping", "believes", "gears", "pried", "later", "ponder"] if w in STORY])

RUN: ../runs/sft_blocks_lr1e-4 | TAG: baseline_blocks
story vocab size: 6683 | SHAKE: 12373
['gears', 'later']


## Baselines

In [27]:
import pandas as pd
from datasets import load_dataset
STORY_FILE = Path(DATA) / "story_vocab.json"                      # cached, so the stream runs once
if STORY_FILE.exists():
    STORY = set(json.load(open(STORY_FILE)))
else:
    counts = Counter()
    for i, ex in enumerate(load_dataset("roneneldan/TinyStories", split="validation", streaming=True)):
        counts.update(words(ex["text"]))
        if i >= 20_000:
            break
    STORY = {w for w, c in counts.items() if c >= 5}
    json.dump(sorted(STORY), open(STORY_FILE, "w"))
print("RUN:", RUN, "| story vocab:", len(STORY), "| SHAKE:", len(SHAKE))
print("probe words found in STORY:", [w for w in ["escaping", "believes", "gears", "pried", "later", "ponder"] if w in STORY])

RUN: ../runs/sft_blocks_lr1e-4 | story vocab: 6683 | SHAKE: 12373
probe words found in STORY: ['gears', 'later']


In [28]:
SETTINGS = {"t10": (1.0, None), "t08k50": (0.8, 50)}              # RL-style sampling vs a sharper decoding setting


def run_baseline(tag, temperature, k):
    rows = []
    for history, partner in CONTEXTS:
        reps = sample_replies(history, partner, temperature=temperature, k=k)
        rows.append({"history": history, "partner": partner, "replies": reps, "metrics": metrics(history[-1][1], reps)})
    mean = {key: round(float(np.mean([r["metrics"][key] for r in rows])), 3) for key in rows[0]["metrics"]}
    Path("../runs/improv").mkdir(parents=True, exist_ok=True)
    json.dump({"run": RUN, "sampling": {"temperature": temperature, "k": k, "n": 8, "max_new_tokens": 60},
               "mean": mean, "rows": rows}, open(f"../runs/improv/{tag}.json", "w"), indent=1)
    return rows, mean


all_rows, means = {}, {}
for name, (temperature, k) in SETTINGS.items():
    all_rows[name], means[name] = run_baseline(f"{TAG}_{name}", temperature, k)
print(pd.DataFrame(means).T.to_string())

for name, rows in all_rows.items():                                  # read the text yourself
    print("=" * 30, name)
    for row in rows[:3]:
        print(row["partner"], "<-", row["history"][-1][1])
        for x in row["replies"][:4]:
            print("  •", x["reply"].replace("\n", " / ") or "(empty)")
    inv = [w for row in rows[:4] for r in row["replies"] for w in words(r["reply"]) if word_class(w) == "invented"]
    print("invented:", sorted(set(inv))[:30])

        stopped   words  overlap  new_words  repeat_bigram  distinct_replies  overlap_frac  shake_words  story_words  invented_words
t10       0.766  15.734    0.234      8.344          0.003               1.0         0.025        0.895        0.004           0.101
t08k50    0.875  12.516    0.188      5.828          0.006               1.0         0.037        0.970        0.002           0.028
============================== t10
JULIET <- Wilt thou not walk with me beneath the moon?
  • Swall thy that raging thy bark, / Yet, marry, say that it is a wooish whore: / I am now two hap banished, and ere I make her dead, / From gallawing liberty, and
  • I am a mystery, I cannot betwhat's against him.
  • Very well to meet in the porining of the stars / Sure conclus your grace's content lord as death / That pray thus to I veot. I wish thus short thread / To meet God R danger! But that die langu
  • I ponder of it: by what weare, what, a kind, that puts / As I dismX shall be upon, the most d

In [29]:
old = json.load(open("../runs/improv/baseline_sft_lr1e-4.json"))["mean"]       # or whatever TAG you used for it
print(pd.DataFrame({"old": old, "new_t10": means["t10"]}).dropna().round(3))

                     old  new_t10
stopped            0.781    0.766
words             14.641   15.734
overlap            0.203    0.234
new_words          7.969    8.344
repeat_bigram      0.004    0.003
distinct_replies   1.000    1.000
shake_words        0.918    0.895
story_words        0.001    0.004
invented_words     0.081    0.101


In [30]:
import json, pandas as pd
rows = {t: json.load(open(f"../runs/improv/baseline_blocks_{t}.json"))["mean"] for t in ["t10", "t08k50"]}
print(pd.DataFrame(rows).T.round(3).to_string())

        stopped   words  overlap  new_words  repeat_bigram  distinct_replies  overlap_frac  shake_words  story_words  invented_words
t10       0.766  15.734    0.234      8.344          0.003               1.0         0.025        0.895        0.004           0.101
t08k50    0.875  12.516    0.188      5.828          0.006               1.0         0.037        0.970        0.002           0.028
